# RevalExo signal adapter audit

Verify the Xsens tracker mapping and sampling before external inference.

In [1]:
from pathlib import Path
import ast
import numpy as np
import pandas as pd
import h5py
ROOT=Path('../data/raw/revalexo').resolve()
OUT=Path('../data/processed/revalexo_adapter_audit.csv')
ORDER=['Pelvis','Right Upper Leg','Right Lower Leg','Right Foot','Left Upper Leg','Left Lower Leg','Left Foot']
rows=[]
for folder in sorted(ROOT.glob('raw_full_part*/raw_full/Subject*')):
    group=folder.name.rsplit('_',1)[-1]
    if group not in {'HC','ST'}: continue
    with h5py.File(folder/'mvn-analyze.hdf5','r') as f:
        g=f['mvn-analyze/xsens-motion-trackers']; accel=g['free_acceleration']; gyro=g['gyroscope']
        headings=ast.literal_eval(''.join(list(accel.attrs['Data headings'])))
        t=g['time_since_start_s'][:].ravel()/1e6; fs=1/np.median(np.diff(t))
        assert headings==ORDER and accel.shape[1]==7 and gyro.shape[1]==7
        rows.append({'subject':folder.name,'group':group,'source_hz':fs,'samples':len(t),'duration_s':t[-1]-t[0],'accel_unit':accel.attrs['Units'],'gyro_unit_metadata':gyro.attrs['Units'],'mapping':'LB=Pelvis;RF=Right Foot;LF=Left Foot','gyro_conversion':'UNRESOLVED'})
audit=pd.DataFrame(rows)
display(audit.groupby('group').agg(subjects=('subject','nunique'),median_source_hz=('source_hz','median'),median_duration_s=('duration_s','median')))
OUT.parent.mkdir(parents=True,exist_ok=True); audit.to_csv(OUT,index=False); print(OUT)

,subjects,median_source_hz,median_duration_s
group,,,
HC,7,58.935253,33567.664062
ST,10,58.935253,38672.484375


..\data\processed\revalexo_adapter_audit.csv
